In [1]:
library(M4comp2018)
library(forecast)
library(dplyr)

Registered S3 method overwritten by 'quantmod':
  method            from
  as.zoo.data.frame zoo 


载入程辑包：‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




In [2]:
Yearly_M4 <- Filter(function(l) l$period == "Yearly", M4)

In [3]:
data=Yearly_M4

In [12]:
pre_forecast_ets=function(data,freq,h,m,n,train_index)
{
    data_length=length(data)
    pred = array(0,dim = c(data_length,m,n,6))
    predh = array(0,dim = c(data_length,m,n,h))
    train_matrix <- matrix(0,ncol = 3, nrow =data_length)
    test_matrix <- matrix(0,ncol = 3, nrow =data_length)
    colnames(train_matrix) <- c("user_time", "system_time", "elapsed_time")
    colnames(test_matrix) <- c("user_time", "system_time", "elapsed_time")
    for(k in 1:data_length){
        start_time = Sys.time()
        y = data[[k]]$x
        y_pred = data[[k]]$xx
        y_l=length(y)
        loc = 1:length(y)
        loc_m = as.integer(loc*m/length(y))
        filt_d = data.frame(y,loc_m)
        filt_0=filter(filt_d ,loc_m==0)
        m_l=count(filt_0)
        n_l=m_l/n
        m_l=as.integer(m_l)
        n_l=as.numeric(n_l)
        for(i in 0:(m-1)){
            for(j in 0:(n-1)){
                Y = ts(y[round((i*m_l+j*n_l)+1):y_l], end = end(y), frequency=freq)
                M  <- tryCatch({
                M = nnetar(Y)    
                }, error = function(e) {
                  # 如果发生错误，则运行ets模型
                  M <- ets(Y)
                  return(M)
                })
                pd=forecast(M, h=h)
                predh[k,(i+1),(j+1),] =pd$mean 
                pred[k,(i+1),(j+1),] = accuracy(pd, y_pred)[2,1:6]
            }
        }
        end_time = Sys.time()
        if (k %in% train_index) {
            train_matrix[k,] = end_time - start_time
        } else {
            test_matrix[k,] = end_time - start_time
        }
    }
    return(list(pred, predh, train_matrix, test_matrix))
}

In [13]:
set.seed(100)
index = sample(2,length(data),replace = TRUE,prob=c(0.7,0.3))
trainindex=c(1:length(data))[index==1]

In [14]:
freq=1
h=6

In [ ]:
datalist=pre_forecast_ets(data,freq,h,5,4,trainindex)

Warning message in nnetar(Y):
“Constant data, setting p=1, P=0, lambda=NULL, scale.inputs=FALSE”
Warning message in nnetar(Y):
“Constant data, setting p=1, P=0, lambda=NULL, scale.inputs=FALSE”
Warning message in nnetar(Y):
“Constant data, setting p=1, P=0, lambda=NULL, scale.inputs=FALSE”


In [16]:
save(datalist, file = "Yearly_nnetar_datalist.RData")